# 03_dl_W120

Conv1D + Transformer, W = 120 s, all seeds (resumes and skips finished seeds).

Code commit `6c08f7948b9e`. Runs on Kaggle and locally (dry-run subset).

In [ ]:
# 03_dl_W120: Conv1D + Transformer, W = 120 s, all seeds (resumes and skips finished seeds).
# Generated by scripts/make_kaggle_notebooks.py; edit templates/, not this file.
NOTEBOOK = '03_dl_W120'
CODE_COMMIT = '6c08f7948b9ecef06a2b223ee24473644a7f0427'
RUN = [['scripts/train_dl.py', '--window', '120', '--samples', '{SAMPLES}', '--prep', '{PREP}', '--work', '{WORK}'], ['scripts/train_dl.py', '--window', '120', '--final-push-only', '--samples', '{SAMPLES}', '--prep', '{PREP}', '--work', '{WORK}']]
SAMPLES_SHA256 = '9b17751ad6fe0874274a5038545d36725f7de4bf0f53ff81e20d8369b83f2d2c'  # sha256 of samples_v2/samples.json that was checked locally
NEEDS_WAVE = True
NEEDS_HF = True  # on Kaggle: stop at once without HF_TOKEN or a real hub.runs_repo
NEEDS_TORCH = True
TORCH_INDEX = 'https://download.pytorch.org/whl/cu126'  # CUDA build of the locked torch version
LOCAL_ARGS = '--subset dryrun --max-samples 2000 --epochs 1 --seeds 1 --num-workers 2'  # local run only: dry-run subset, never pushes
LOCAL_INPUT_NOTEBOOKS = []

In [ ]:
# Runtime, inputs, integrity checks and paths. The same notebook runs on Kaggle and locally.
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT = Path(os.environ.get("UC04_KAGGLE_INPUT", "/kaggle/input"))       # overridable for tests only
KAGGLE_WORKING = Path(os.environ.get("UC04_KAGGLE_WORKING", "/kaggle/working"))
RUNTIME = os.environ.get("UC04_RUNTIME_OVERRIDE") or ("kaggle" if KAGGLE_INPUT.exists() else "local")
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["UC04_RUNTIME"] = RUNTIME  # on Kaggle the scripts refuse to run without HF (results would be lost)


def find_dirs(root, marker, want_dir=None):
    """Directories under `root` containing `marker` (and sub-directory `want_dir` if given)."""
    out = sorted({p.parent for p in Path(root).rglob(marker)})
    return [d for d in out if want_dir is None or (d / want_dir).is_dir()]


def one(dirs, what):
    """Exactly one match, otherwise stop (never guess between several attached datasets)."""
    if len(dirs) != 1:
        raise RuntimeError(f"expected exactly one {what} in the notebook inputs, found {len(dirs)}: {dirs}. "
                           "Check Add Input and !ls " + str(KAGGLE_INPUT))
    return dirs[0]


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


if RUNTIME == "kaggle":
    code_src = one(find_dirs(KAGGLE_INPUT, "CODE_COMMIT"), "uc04-code dataset")
    REPO = KAGGLE_WORKING / "uc04"
    shutil.copytree(code_src, REPO, dirs_exist_ok=True)
    got = (REPO / "CODE_COMMIT").read_text().strip()
    if got != CODE_COMMIT:
        raise RuntimeError(f"uc04-code dataset is commit {got[:12]}, this notebook was generated for {CODE_COMMIT[:12]}")
    cfg = json.loads((REPO / "configs" / "uc04_v2.json").read_text(encoding="utf-8"))
    SAMPLES = one(find_dirs(KAGGLE_INPUT, "samples.json"), "uc04-samples-v2 dataset")
    PREP = one(find_dirs(KAGGLE_INPUT, "qc.csv", want_dir="wave100"), "wave dataset (uc04-prep-v1)") if NEEDS_WAVE else None
    INPUTS = []  # results of earlier notebooks come from Hugging Face, never from kernel outputs
    WORK = KAGGLE_WORKING / "work"
    EXTRA = []
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "uc04_v2.json").exists())
    cfg = json.loads((REPO / "configs" / "uc04_v2.json").read_text(encoding="utf-8"))
    SAMPLES = REPO / cfg["paths"]["samples"]
    PREP = Path(cfg["paths"]["prep"]) if NEEDS_WAVE else None
    INPUTS = [REPO / "artifacts_dryrun" / f"nb_{n}" for n in LOCAL_INPUT_NOTEBOOKS]
    WORK = REPO / "artifacts_dryrun" / f"nb_{NOTEBOOK}"
    EXTRA = LOCAL_ARGS.split()

# integrity: the exact samples and wave data that were checked locally
got = sha256_of(Path(SAMPLES) / "samples.json")
if got != SAMPLES_SHA256:
    raise RuntimeError(f"samples.json sha256 {got[:12]} != expected {SAMPLES_SHA256[:12]}: wrong uc04-samples-v2 version")
if NEEDS_WAVE:
    got = sha256_of(Path(PREP) / "qc.csv")
    if got != cfg["input"]["qc_sha256"]:
        raise RuntimeError(f"qc.csv sha256 {got[:12]} != input.qc_sha256 {cfg['input']['qc_sha256'][:12]}: "
                           "wrong uc04-prep-v1 dataset")

# Hugging Face is required on Kaggle for notebooks that write results (a stopped session may lose its output)
if RUNTIME == "kaggle" and NEEDS_HF:
    if "<" in cfg["hub"]["runs_repo"]:
        raise RuntimeError("hub.runs_repo in configs/uc04_v2.json is still a placeholder: create a private HF repo, "
                           "set it in the config, commit, and rebuild uc04-code")
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(cfg["hub"]["token_env"])
    except Exception:
        raise RuntimeError("Kaggle Secret HF_TOKEN is missing or not enabled for this notebook "
                           "(Add-ons -> Secrets). Training on Kaggle requires HF.") from None
    if not token:
        raise RuntimeError("Kaggle Secret HF_TOKEN is empty")
    os.environ[cfg["hub"]["token_env"]] = token  # never printed
    del token
    print("HF_TOKEN secret found; runs_repo =", cfg["hub"]["runs_repo"])

WORK.mkdir(parents=True, exist_ok=True)
print(f"RUNTIME={RUNTIME}\nREPO={REPO}\nSAMPLES={SAMPLES}\nPREP={PREP}\nWORK={WORK}\nINPUTS={INPUTS}\nEXTRA={EXTRA}")
print("checks passed: " + ", ".join((["code commit"] if RUNTIME == "kaggle" else []) + ["samples.json sha256"]
                                  + (["qc.csv sha256"] if NEEDS_WAVE else [])))

In [ ]:
# Install the locked library versions (Kaggle only; the local venv is the lock).
def sh(*cmd):
    """Run a command and stream its output into the notebook; raise on failure."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


if RUNTIME == "kaggle":
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", REPO / "requirements-lock.txt")
    if NEEDS_TORCH:
        torch_version = json.loads((REPO / "requirements-lock.json").read_text())["torch_base_version"]
        sh(sys.executable, "-m", "pip", "install", "-q", f"torch=={torch_version}", "--index-url", TORCH_INDEX)
    sh(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO)

In [ ]:
# Library versions must match requirements-lock.json (model-critical packages).
sh(sys.executable, REPO / "scripts" / "env_check.py", *([] if NEEDS_TORCH else ["--ignore", "torch"]))
if NEEDS_TORCH:
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available(), "
                        "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
                       capture_output=True, text=True)
    print("torch:", r.stdout.strip() or r.stderr.strip())

In [ ]:
# Run the step. Each script writes only its own files and skips work already done.
def fill(arg):
    return {"{SAMPLES}": SAMPLES, "{WORK}": WORK, "{PREP}": PREP}.get(arg, arg)


for step in RUN:
    args = [fill(a) for a in step]
    if "{INPUTS}" in args:
        i = args.index("{INPUTS}")
        args = args[:i] + [str(p) for p in INPUTS] + args[i + 1:]
    sh(sys.executable, REPO / args[0], *args[1:], *EXTRA)

In [ ]:
# What this notebook produced.
files = sorted(p for p in WORK.rglob("*") if p.is_file())
print(f"{len(files)} files in {WORK}")
for p in files[:40]:
    print("  ", p.relative_to(WORK))
if len(files) > 40:
    print(f"   ... and {len(files) - 40} more")